# Export Data for Figure_S_pct_forest_burned.jpg
Processes raw inputs and saves all data needed for the figure to `../figure_data/figure_S_pct_forest_burned/`.

In [ ]:
import os

import xarray as xr

from western_us_biomass import dir_info
from western_us_biomass.make_figures import maps

OUT_DIR = "../figure_data/figure_S_pct_forest_burned/"
os.makedirs(OUT_DIR, exist_ok=True)

## Load inputs and compute derived variables

In [2]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")

western_states = maps.SHP_WESTERN.to_crs(crs)

# Forest mask
is_forest = inputs["is_forest"].sel(year=2021).rio.write_crs(crs)
is_forest_clipped = is_forest.rio.clip(western_states.geometry)

# Recent burn flag
years_after_fire = inputs["years_after_fire"].sel(year=2022).rio.write_crs(crs)
years_after_fire_clipped = years_after_fire.rio.clip(western_states.geometry)
fire_year = (2022 - years_after_fire_clipped).where(years_after_fire_clipped < 80)
recent_burn = fire_year >= 2015

# Climate variables masked to forest
tmean = inputs["tmean_clim_mean"].rio.write_crs(crs).rio.clip(western_states.geometry)
ppt = inputs["ppt_clim_mean"].rio.write_crs(crs).rio.clip(western_states.geometry)

tmean_forest = tmean.where(is_forest_clipped > 50)
ppt_forest = ppt.where(is_forest_clipped > 50)

## Save rasters

In [3]:
tmean_forest.to_dataset(name="tmean_forest").to_netcdf(OUT_DIR + "tmean_forest.nc")
print("Saved tmean_forest.nc")

ppt_forest.to_dataset(name="ppt_forest").to_netcdf(OUT_DIR + "ppt_forest.nc")
print("Saved ppt_forest.nc")

recent_burn.to_dataset(name="recent_burn").to_netcdf(OUT_DIR + "recent_burn.nc")
print("Saved recent_burn.nc")

Saved tmean_forest.nc
Saved ppt_forest.nc
Saved recent_burn.nc


## Save state boundary shapefiles

In [4]:
CA_boundary = western_states[western_states["STUSPS"] == "CA"]
PNW_boundary = western_states[western_states["STUSPS"].isin(["WA", "OR"])]

CA_boundary.to_file(OUT_DIR + "CA_boundary.gpkg", driver="GPKG")
print("Saved CA_boundary.gpkg")

PNW_boundary.to_file(OUT_DIR + "PNW_boundary.gpkg", driver="GPKG")
print("Saved PNW_boundary.gpkg")

Saved CA_boundary.gpkg
Saved PNW_boundary.gpkg
